In [41]:
import numpy as np 
import pandas as pd 
import seaborn as sns 
from matplotlib import pyplot as plt 

In [2]:
df = pd.read_csv("play_tennis.csv")
df.head()

,day,outlook,temp,humidity,wind,play
0,D1,Sunny,Hot,High,Weak,No
1,D2,Sunny,Hot,High,Strong,No
2,D3,Overcast,Hot,High,Weak,Yes
3,D4,Rain,Mild,High,Weak,Yes
4,D5,Rain,Cool,Normal,Weak,Yes


In [3]:
df.drop(columns = ['day'] , axis = 1 , inplace = True)

In [4]:
df.head()

,outlook,temp,humidity,wind,play
0,Sunny,Hot,High,Weak,No
1,Sunny,Hot,High,Strong,No
2,Overcast,Hot,High,Weak,Yes
3,Rain,Mild,High,Weak,Yes
4,Rain,Cool,Normal,Weak,Yes


### `problem statement-01:`
---
     given outlook = Sunny , temp = Hot , humidity = Weak => Find play Yes or NO?
        Solution:
                   P1 = P(Yes | Sunny , Hot , Weak) = P(Sunny|Yes) * P(Hot|yes) * P(Weak|Yes).
                   P2 = P(No | Sunny , Hot , Weak) = P(Sunny|No) * P(Hot|No) * P(Weak|No). 
                   If P1 > P2 ==> Play yes otherwise No.
--- 

### `problem statement-02:`
---
     given outlook = Overcast , temp = Cool , humidity = Weak => Find play Yes or NO?
        Solution:
                   P1 = P(Yes | Overcast , Cool , Weak) = P(Overcast|Yes) * P(Cool|yes) * P(Weak|Yes).
                   P2 = P(No | Sunny , Cool , Weak) = P(Overcast|No) * P(Cool|No) * P(Weak|No). 
                   If P1 > P2 ==> Play yes otherwise No.

- As for different different input features we have to compute different different probabilities. So naive bayes do in training stage calculate the probability for each unique values of every input features.
And stores these result in a Look-Up table like a `Dict` type data structures. 

- And at test phase it just collect these probabilities from the table and do the predictions.

In [5]:
df['play'].value_counts()

play
Yes    9
No     5
Name: count, dtype: int64

In [6]:
# step-01: first the probability of all class label's 
prob_yes = 9 / (9 + 5)
prob_no = 5 / (9 + 5)

print("prob_yes:" , prob_yes) 
print("prob_no:" , prob_no)

prob_yes: 0.6428571428571429
prob_no: 0.35714285714285715


#### step-2: go to every column and find the probability for each unique values of this column.

In [7]:
# outlook column 
df['outlook'].value_counts

outlook
Sunny       5
Rain        5
Overcast    4
Name: count, dtype: int64

In [11]:
table = pd.crosstab(df['outlook'] , df['play'])
table

play,No,Yes
outlook,,
Overcast,0,4
Rain,2,3
Sunny,3,2


`Note`: There is an zero value for Overcast. So it's probability will be zero. And for every result where Overcast will be present then result will be `Yes` no matter what are the other features. That's called `Zero probability Issue` of Naive Bayes. So to avoid this what we do is add 1 with every value of this column where zero is present to avoid this issue. This is called `Laplace Smoothing`.

Now if we add 1 to a single column then it also change the class count. So when we will calculate overall probability at that time , the dinominator will be different for outlook. Means we are treating this feature differently, which breaks the consistency of Naive Bayes. so to maintain the consistency what we do is add 1 to the frequency of every feature's value counts. 

In [13]:
smoothed = table + 1
smoothed

play,No,Yes
outlook,,
Overcast,1,5
Rain,3,4
Sunny,4,3


In [14]:
prob_overcast_yes = 5/12
prob_overcast_no = 1/8

In [18]:
prob_rain_yes = 4/12
prob_rain_no = 3/8

In [19]:
prob_sunny_yes = 3/12
prob_sunny_no = 4/8

In [20]:
# temp column 
df['temp'].value_counts()

temp
Mild    6
Hot     4
Cool    4
Name: count, dtype: int64

In [21]:
table = pd.crosstab(df['temp'] , df['play'])
table

play,No,Yes
temp,,
Cool,1,3
Hot,2,2
Mild,2,4


In [22]:
smoothed = table + 1 # 1 need to be added for outlook -> overcast
smoothed

play,No,Yes
temp,,
Cool,2,4
Hot,3,3
Mild,3,5


In [23]:
prob_cool_yes = 4/12
prob_cool_no = 2/8

prob_hot_yes = 3/12
prob_hot_no = 3/8

prob_mild_yes = 5/12
prob_mild_no = 3/8

In [24]:
# humidity column 
df['humidity'].value_counts()

humidity
High      7
Normal    7
Name: count, dtype: int64

In [25]:
table = pd.crosstab(df['humidity'] , df['play'])
table

play,No,Yes
humidity,,
High,4,3
Normal,1,6


In [26]:
smoothed = table + 1 # 1 need to be added for outlook -> overcast
smoothed

play,No,Yes
humidity,,
High,5,4
Normal,2,7


In [29]:
prob_high_yes = 4 / 11
prob_high_no  = 5 / 7

prob_normal_yes = 7/11
prob_normal_no = 2/7

In [30]:
# wind column 
df['wind'].value_counts()

wind
Weak      8
Strong    6
Name: count, dtype: int64

In [31]:
table = pd.crosstab(df['wind'] , df['play'])
table

play,No,Yes
wind,,
Strong,3,3
Weak,2,6


In [32]:
smoothed = table + 1

In [33]:
smoothed

play,No,Yes
wind,,
Strong,4,4
Weak,3,7


In [34]:
prob_strong_no = 4/7
prob_strong_yes = 4/11

prob_weak_no = 3/7
prob_weak_yes = 7/11

In [35]:
# problem-01: outlook = Sunny , temp = Hot , humidity = Weak => Find play Yes or NO?
p1 = prob_sunny_yes * prob_hot_yes * prob_weak_yes
p2 = prob_sunny_no * prob_hot_no * prob_weak_no

print("p1:" , p1)
print("p2:" , p2)

p1: 0.03977272727272727
p2: 0.08035714285714285


In [36]:
play = "Yes" if p1 > p2 else "No"
play

'No'

In [37]:
# problem-02: outlook = Overcast , temp = Cool , humidity = Weak => Find play Yes or NO?
p1 = prob_overcast_yes * prob_cool_yes * prob_weak_yes
p2 = prob_overcast_no * prob_cool_no * prob_weak_no

print("p1:" , p1)
print("p2:" , p2)

p1: 0.08838383838383838
p2: 0.013392857142857142


In [38]:
play = "Yes" if p1 > p2 else "No"
play

'Yes'

------------------------------------------------- `How Naive Bayes Handle Numerical Values` ---------------------------------------------------

In [39]:
data = {
    'Feature1': [5.1, 4.9, 6.2, 5.8, 7.1, 6.5, 5.5, 6.0],
    'Feature2': [3.5, 3.0, 3.4, 2.8, 3.0, 3.2, 3.6, 3.1],
    'Feature3': [1.4, 1.5, 5.4, 4.5, 5.9, 5.7, 1.3, 4.8],
    'Class':    ['Yes', 'Yes', 'No', 'No', 'No', 'No', 'Yes', 'No']
}
df1 = pd.DataFrame(data)

In [40]:
df1.head()

,Feature1,Feature2,Feature3,Class
0,5.1,3.5,1.4,Yes
1,4.9,3.0,1.5,Yes
2,6.2,3.4,5.4,No
3,5.8,2.8,4.5,No
4,7.1,3.0,5.9,No


In [52]:
class_counts = df1['Class'].value_counts().sort_index()  

In [53]:
class_counts

Class
No     5
Yes    3
Name: count, dtype: int64

In [55]:
# find the probability of each class 
total = len(df1)
priors = (class_counts / total).to_dict()
priors

{'No': 0.625, 'Yes': 0.375}

In [60]:
# now find mean and std for each feature per class 
grouped = df1.groupby('Class').agg(
    Feature1_mean = ('Feature1', 'mean'),
    Feature1_std = ('Feature1', lambda x: x.std(ddof = 0)),
    Feature2_mean = ('Feature2', 'mean'),
    Feature2_std = ('Feature2', lambda x: x.std(ddof = 0)),
    Feature3_mean = ('Feature3', 'mean'),
    Feature3_std = ('Feature3', lambda x: x.std(ddof = 0))
)

In [61]:
grouped

,Feature1_mean,Feature1_std,Feature2_mean,Feature2_std,Feature3_mean,Feature3_std
Class,,,,,,
No,6.320000,0.453431,3.100000,0.200000,5.26,0.531413
Yes,5.166667,0.249444,3.366667,0.262467,1.40,0.081650


In [62]:
import math
# find the gaussian pdf 
def gaussian_pdf(x , mu , sigma): 
    if sigma == 0: 
        return 1.0 if x == 0 else 1e-9

    coef = 1 / (sigma * math.sqrt(2 * math.pi))
    exponent = math.exp(-0.5 * ((x - mu) / sigma) ** 2)
    return coef * exponent

In [63]:
# now let's say we have a sample that we have to predict 
x = [6.0, 3.0, 5.0] 

In [67]:
# For class = Yes
pdf_f1_yes = gaussian_pdf(x[0], mu=5.166667, sigma=0.249444)
pdf_f2_yes = gaussian_pdf(x[1], mu=3.366667, sigma=0.262467)
pdf_f3_yes = gaussian_pdf(x[2], mu=1.40, sigma=0.081650)

p_yes = pdf_f1_yes * pdf_f2_yes * pdf_f3_yes

In [68]:
# For class = No
pdf_f1_no = gaussian_pdf(x[0], mu=6.320000, sigma=0.453431)
pdf_f2_no = gaussian_pdf(x[1], mu=3.100000, sigma=0.200000)
pdf_f3_no = gaussian_pdf(x[2], mu=5.26, sigma=0.531413)

p_no = pdf_f1_no * pdf_f2_no * pdf_f3_no

In [69]:
(p1 , p2)

(0.804152903946222, 0.013392857142857142)

In [70]:
# now calculate p(Yes | x) and p(No | x) 
posterior_yes = p_yes * priors['Yes']
posterior_no = p_no * priors['No']

In [71]:
if posterior_yes > posterior_no:
    prediction = "Yes"
else:
    prediction = "No"

prediction

'No'